# Netflix Data Cleaning
### Handling Missing Values and Datetime Data with Pandas

**Name:** Mark Yosinao
**Date:** November 5, 2026

In [28]:
import pandas as pd

netflix_titles = pd.read_csv("netflix_titles.csv")
netflix_titles.shape

(8807, 12)

In [29]:
netflix_titles.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [30]:
(netflix_titles.isnull().mean() * 100).round(2)

show_id          0.00
type             0.00
title            0.00
director        29.91
cast             9.37
country          9.44
date_added       0.11
release_year     0.00
rating           0.05
duration         0.03
listed_in        0.00
description      0.00
dtype: float64

### Missing Data Summary
The `director` column has the worst data gap, with about 29.9% of values missing.
`country` (9.4%) and `cast` (9.4%) are next. The remaining columns, such as
`date_added`, `rating`, and `duration`, are missing less than 1% of their values.

In [31]:
experimental_df = netflix_titles.copy()

### Strategy A: Imputation

In [32]:
netflix_titles["director"] = netflix_titles["director"].fillna("Unknown")
netflix_titles["director"].isnull().sum()

np.int64(0)

### Strategy B: Row Dropping

In [33]:
experimental_df = experimental_df.dropna(subset=["director", "cast"])
print("Original shape:", netflix_titles.shape)
print("After dropping:", experimental_df.shape)
print("Rows lost:", len(netflix_titles) - len(experimental_df))

Original shape: (8807, 12)
After dropping: (5700, 12)
Rows lost: 3107


### Reflection
After dropping rows with missing `director` or `cast`, the DataFrame went from
8,807 rows to 5,700 rows, so 3,107 rows were lost (about 35.3% of the data).

Losing this much data could ruin an analysis that compares Movies and TV Shows.
TV shows often don't list a director, so dropping those rows would remove most of
the TV shows. Any conclusion about what Netflix offers would then be biased toward
movies and wouldn't reflect the full catalog.

In [34]:
netflix_titles.dtypes

show_id           str
type              str
title             str
director          str
cast              str
country           str
date_added        str
release_year    int64
rating            str
duration          str
listed_in         str
description       str
dtype: object

Note: `date_added` shows as `str` because this project uses pandas 3, which labels
text columns as `str` instead of `object`. Either way, it is stored as text, not a date.

In [35]:
netflix_titles["date_added"] = pd.to_datetime(
    netflix_titles["date_added"].str.strip(), format="%B %d, %Y"
)
netflix_titles.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8807 non-null   str           
 1   type          8807 non-null   str           
 2   title         8807 non-null   str           
 3   director      8807 non-null   str           
 4   cast          7982 non-null   str           
 5   country       7976 non-null   str           
 6   date_added    8797 non-null   datetime64[us]
 7   release_year  8807 non-null   int64         
 8   rating        8803 non-null   str           
 9   duration      8804 non-null   str           
 10  listed_in     8807 non-null   str           
 11  description   8807 non-null   str           
dtypes: datetime64[us](1), int64(1), str(10)
memory usage: 825.8 KB


In [36]:
netflix_titles["day_added"] = netflix_titles["date_added"].dt.day
netflix_titles.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,day_added
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,2021-09-25,2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm...",25.0
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t...",24.0
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...,24.0
3,s4,TV Show,Jailbirds New Orleans,Unknown,NaN,NaN,2021-09-24,2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo...",24.0
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...,24.0


## Conclusion
- The `director` column had the most missing data at about 29.9%.
- Filling missing directors with "Unknown" kept all 8,807 rows.
- Dropping rows with missing `director` or `cast` removed 3,107 rows (35.3%).
- Imputation is often the safer choice when a lot of data is missing.
- Converting `date_added` to datetime made it possible to extract the day each title was added.